In [5]:
import pandas as pd

# =========================================================
# ABLE-BAKER QUEUE SIMULATION
# CONDITION: ABLE IS FASTER THAN BAKER
# =========================================================

# ---------------------------------------------------------
# 1. CUSTOMER INTER-ARRIVAL TIME DISTRIBUTION
# ---------------------------------------------------------
inter_arrival_times = [1, 2, 3, 4, 5]
inter_arrival_probs = [0.25, 0.40, 0.15, 0.10, 0.10]

# ---------------------------------------------------------
# 2. ABLE SERVICE TIME DISTRIBUTION (FASTER)
# ---------------------------------------------------------
able_service_times = [1, 2, 3, 4]
able_service_probs = [0.30, 0.28, 0.25, 0.17]

# ---------------------------------------------------------
# 3. BAKER SERVICE TIME DISTRIBUTION (SLOWER)
# ---------------------------------------------------------
baker_service_times = [2, 3, 4, 5, 6]
baker_service_probs = [0.25, 0.25, 0.20, 0.20, 0.10]


# =========================================================
# FUNCTION TO CREATE RDA TABLE
# =========================================================
def create_rda_table(values, probabilities, column_name):

    cumulative_probs = []
    total = 0

    for p in probabilities:
        total += p
        cumulative_probs.append(round(total, 2))

    ranges = []
    start = 0

    for cp in cumulative_probs:

        end = round(cp * 100)

        if end == 100:
            ranges.append(f"{start:02d}-{end:03d}")
        else:
            ranges.append(f"{start:02d}-{end:02d}")

        start = end + 1

    return pd.DataFrame({
        column_name: values,
        'Probability': probabilities,
        'Cumulative Probability': cumulative_probs,
        'Random Digit Range': ranges
    })


# =========================================================
# CREATE RDA TABLES
# =========================================================
inter_arrival_table = create_rda_table(
    inter_arrival_times,
    inter_arrival_probs,
    'Inter-Arrival Time'
)

able_table = create_rda_table(
    able_service_times,
    able_service_probs,
    'Able Service Time'
)

baker_table = create_rda_table(
    baker_service_times,
    baker_service_probs,
    'Baker Service Time'
)


# =========================================================
# FUNCTION TO GET TIME FROM RANDOM DIGIT
# =========================================================
def get_time(random_digit, rda_table, column_name):

    for _, row in rda_table.iterrows():

        start_str, end_str = row['Random Digit Range'].split('-')

        start = int(start_str)
        end = int(end_str)

        if start <= random_digit <= end:
            return row[column_name]

    return None


# =========================================================
# RANDOM DIGITS
# =========================================================

arrival_random_digits = [0, 22, 76, 89, 65, 26, 42]

able_random_digits = [23, 59, 21, 51, 82, 89, 48]

baker_random_digits = [23, 59, 21, 51, 82, 89, 48]


# =========================================================
# GENERATE INTER-ARRIVAL TIMES
# =========================================================
inter_arrival_generated = []

for rd in arrival_random_digits:

    if rd == 0:
        inter_arrival_generated.append(0)

    else:
        time = get_time(
            rd,
            inter_arrival_table,
            'Inter-Arrival Time'
        )

        inter_arrival_generated.append(time)


# =========================================================
# SIMULATION
# =========================================================

arrival_times = []
server_assigned = []
service_times = []
service_begin = []
service_end = []
waiting_times = []
time_in_system = []
able_idle_times = []
baker_idle_times = []

able_available_time = 0
baker_available_time = 0

current_arrival = 0

for i in range(len(arrival_random_digits)):

    # -----------------------------------------------------
    # ARRIVAL TIME
    # -----------------------------------------------------
    current_arrival += inter_arrival_generated[i]

    arrival_times.append(current_arrival)

    # -----------------------------------------------------
    # SERVER ASSIGNMENT LOGIC
    # CONDITION:
    # IF BOTH ARE FREE, ASSIGN TO ABLE
    # SINCE ABLE IS FASTER
    # -----------------------------------------------------

    if current_arrival >= able_available_time and current_arrival >= baker_available_time:

        server = "Able"

    elif able_available_time <= baker_available_time:

        server = "Able"

    else:

        server = "Baker"

    # -----------------------------------------------------
    # ABLE SERVER
    # -----------------------------------------------------
    if server == "Able":

        rd_service = able_random_digits[i]

        service_time = get_time(
            rd_service,
            able_table,
            'Able Service Time'
        )

        begin_time = max(current_arrival, able_available_time)

        end_time = begin_time + service_time

        wait_time = begin_time - current_arrival

        total_time = end_time - current_arrival

        idle_able = max(0, begin_time - able_available_time)

        idle_baker = 0

        able_available_time = end_time

    # -----------------------------------------------------
    # BAKER SERVER
    # -----------------------------------------------------
    else:

        rd_service = baker_random_digits[i]

        service_time = get_time(
            rd_service,
            baker_table,
            'Baker Service Time'
        )

        begin_time = max(current_arrival, baker_available_time)

        end_time = begin_time + service_time

        wait_time = begin_time - current_arrival

        total_time = end_time - current_arrival

        idle_baker = max(0, begin_time - baker_available_time)

        idle_able = 0

        baker_available_time = end_time

    # -----------------------------------------------------
    # STORE RESULTS
    # -----------------------------------------------------
    server_assigned.append(server)

    service_times.append(service_time)

    service_begin.append(begin_time)

    service_end.append(end_time)

    waiting_times.append(wait_time)

    time_in_system.append(total_time)

    able_idle_times.append(idle_able)

    baker_idle_times.append(idle_baker)


# =========================================================
# FINAL SIMULATION TABLE
# =========================================================
simulation_df = pd.DataFrame({

    'Customer': range(1, len(arrival_random_digits) + 1),

    'Arrival Random Digit': arrival_random_digits,

    'Inter-Arrival Time': inter_arrival_generated,

    'Arrival Time': arrival_times,

    'Server Assigned': server_assigned,

    'Service Time': service_times,

    'Service Begin Time': service_begin,

    'Service End Time': service_end,

    'Waiting Time': waiting_times,

    'Time in System': time_in_system,

    'Able Idle Time': able_idle_times,

    'Baker Idle Time': baker_idle_times
})


# =========================================================
# DISPLAY OUTPUT TABLES
# =========================================================

print("\nINTER-ARRIVAL RDA TABLE\n")
display(inter_arrival_table)

print("\nABLE SERVICE RDA TABLE (FASTER SERVER)\n")
display(able_table)

print("\nBAKER SERVICE RDA TABLE (SLOWER SERVER)\n")
display(baker_table)

print("\nFINAL ABLE-BAKER SIMULATION TABLE\n")
display(simulation_df)


INTER-ARRIVAL RDA TABLE



,Inter-Arrival Time,Probability,Cumulative Probability,Random Digit Range
0,1,0.25,0.25,00-25
1,2,0.40,0.65,26-65
2,3,0.15,0.80,66-80
3,4,0.10,0.90,81-90
4,5,0.10,1.00,91-100



ABLE SERVICE RDA TABLE (FASTER SERVER)



,Able Service Time,Probability,Cumulative Probability,Random Digit Range
0,1,0.30,0.30,00-30
1,2,0.28,0.58,31-58
2,3,0.25,0.83,59-83
3,4,0.17,1.00,84-100



BAKER SERVICE RDA TABLE (SLOWER SERVER)



,Baker Service Time,Probability,Cumulative Probability,Random Digit Range
0,2,0.25,0.25,00-25
1,3,0.25,0.50,26-50
2,4,0.20,0.70,51-70
3,5,0.20,0.90,71-90
4,6,0.10,1.00,91-100



FINAL ABLE-BAKER SIMULATION TABLE



,Customer,Arrival Random Digit,Inter-Arrival Time,Arrival Time,Server Assigned,Service Time,Service Begin Time,Service End Time,Waiting Time,Time in System,Able Idle Time,Baker Idle Time
0,1,0,0,0,Able,1,0,1,0,1,0,0
1,2,22,1,1,Able,3,1,4,0,3,0,0
2,3,76,3,4,Able,1,4,5,0,1,0,0
3,4,89,4,8,Able,2,8,10,0,2,3,0
4,5,65,2,10,Able,3,10,13,0,3,0,0
5,6,26,2,12,Baker,5,12,17,0,5,0,12
6,7,42,2,14,Able,2,14,16,0,2,1,0
